# Final Qdrant Ingestion Pipeline
This notebook loads the mental health dataset, cleans it, generates embeddings using `all-MiniLM-L6-v2`, and uploads the final vectors to Qdrant Cloud. Run this in Google Colab with a GPU.

In [1]:
!pip install sentence-transformers qdrant-client pandas datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 9.2 MB/s eta 0:00:00


In [2]:
import pandas as pd
import re
import uuid
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from google.colab import userdata

In [3]:
# 1. Load Dataset
print("Loading dataset...")
dataset = load_dataset("Amod/mental_health_counseling_conversations")
df = dataset["train"].to_pandas()
print(f"Loaded {len(df)} raw conversations.")

Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

combined_dataset.json:   0%|          | 0.00/4.79M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3512 [00:00<?, ? examples/s]

Loaded 3512 raw conversations.


In [4]:
# 2. Preprocessing & Cleaning
print("Cleaning data...")
df = df.dropna().reset_index(drop=True)
df = df.drop_duplicates().reset_index(drop=True)

def clean_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["Context"] = df["Context"].apply(clean_text)
df["Response"] = df["Response"].apply(clean_text)
print(f"Final cleaned dataset size: {len(df)}")

Cleaning data...
Final cleaned dataset size: 2752


In [5]:
# 3. Initialize Model
print("Loading all-MiniLM-L6-v2...")
model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

Loading all-MiniLM-L6-v2...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
# 4. Generate Embeddings
print("Encoding Contexts...")
contexts = df["Context"].tolist()
responses = df["Response"].tolist()

# all-MiniLM-L6-v2 is extremely fast and light, batch_size=128 is safe
embeddings = model.encode(contexts, batch_size=128, show_progress_bar=True)
print("Embeddings generated successfully.")

Encoding Contexts...


Batches:   0%|          | 0/22 [00:00<?, ?it/s]

Embeddings generated successfully.


In [7]:
# 5. Initialize Qdrant
# Add QDRANT_URL and QDRANT_API_KEY to your Colab Secrets
qdrant_url = userdata.get("QDRANT_URL")
qdrant_api_key = userdata.get("QDRANT_API_KEY")

client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key)
COLLECTION_NAME = "mental_health_counseling"

# Create Collection (Drop if exists for a fresh start)
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)
print(f"Fresh collection \"{COLLECTION_NAME}\" created in Qdrant Cloud.")

Fresh collection "mental_health_counseling" created in Qdrant Cloud.


In [8]:
# 6. Upload to Qdrant
print("Uploading points to Qdrant Cloud...")
points = [
    PointStruct(
        id=str(uuid.uuid4()),
        vector=embedding.tolist(),
        payload={
            "context": context,
            "response": response
        }
    )
    for context, response, embedding in zip(contexts, responses, embeddings)
]

# Upload in batches of 500
client.upload_points(
    collection_name=COLLECTION_NAME,
    points=points,
    batch_size=500
)
print("Upload complete! Your vector database is ready.")

Uploading points to Qdrant Cloud...
Upload complete! Your vector database is ready.
